In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class InceptionModule(nn.Module):
    def __init__(
        self, 
        in_channels, 
        ch1x1, 
        ch3x3red, 
        ch3x3, 
        ch5x5red, 
        ch5x5, 
        pool_proj,
        ):
        super(InceptionModule, self).__init__()
        # x: 192, 28, 28

        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, ch1x1, kernel_size=1),
            nn.ReLU(True) # 64, 28, 28
        )

        # 分支2：1x1 卷积降维 + 3x3 卷积
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, ch3x3red, kernel_size=1),
            nn.ReLU(True), # 96, 28, 28
            nn.Conv2d(ch3x3red, ch3x3, kernel_size=3, padding=1), 
            nn.ReLU(True) # 128, 28, 28
        )

        # 分支3：1x1 卷积降维 + 5x5 卷积
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, ch5x5red, kernel_size=1),
            nn.ReLU(True), # 16, 28, 28
            nn.Conv2d(ch5x5red, ch5x5, kernel_size=5, padding=2), 
            nn.ReLU(True) # 32, 28, 28
        )

        # 分支4：3x3 最大池化 + 1x1 卷积降维
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1), # 32, 28, 28
            nn.Conv2d(in_channels, pool_proj, kernel_size=1), # 32, 28, 28
            nn.ReLU(True)
        )

    def forward(self, x):
        # 分别计算四个分支
        branch1_out = self.branch1(x)
        branch2_out = self.branch2(x)
        branch3_out = self.branch3(x)
        branch4_out = self.branch4(x)
        
        # 在通道维度(dim=1)上进行拼接
        outputs = [branch1_out, branch2_out, branch3_out, branch4_out]
        return torch.cat(outputs, 1) # 64+128+32+32, 28, 28
